**Practical:- 4 Build RNN using Pytorch**

In [ ]:
!pip install torch torchtext pandas scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

deep learning and machine learning framework used to build and train neural networks

In [ ]:
# accessing csv from drive
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Colab Notebooks/Files/IMDB Dataset.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv(file_path)
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


**Encode Labels**

In [ ]:
# The 'sentiment' column was missing, so we create it from 'review_rating'.
# Ratings of 7 and above are treated as positive (1), others as negative (0).
if 'review_rating' in df.columns:
    df['sentiment'] = (df['review_rating'] >= 7).astype(int)
    print("Successfully created 'sentiment' column from 'review_rating'.")
else:
    print(f"Error: Required columns not found. Available columns: {df.columns.tolist()}")

df.head()

Error: Required columns not found. Available columns: ['review', 'sentiment']


,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
# Text Cleaning
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text

df['review'] = df['review'].apply(clean_text)
print(df['review'])

# Build Vocabulary
words = " ".join(df['review']).split()
vocab = Counter(words)

vocab = dict(vocab.most_common(10000))

word_to_idx = {word: i+1 for i, word in enumerate(vocab)}

# Convert Text to Sequences
def encode_review(review):
    return [word_to_idx.get(word, 0) for word in review.split()]

df['encoded'] = df['review'].apply(encode_review)
print(df['encoded'])

# Padding Sequences
MAX_LEN = 200

def pad_sequence(seq):
    if len(seq) > MAX_LEN:
        return seq[:MAX_LEN]
    return seq + [0] * (MAX_LEN - len(seq))

df['padded'] = df['encoded'].apply(pad_sequence)

0        one of the other reviewers has mentioned that ...
1        a wonderful little production the filming tech...
2        i thought this was a wonderful way to spend ti...
3        basically theres a family where a little boy j...
4        petter matteis love in the time of money is a ...
                               ...                        
49995    i thought this movie did a down right good job...
49996    bad plot bad dialogue bad acting idiotic direc...
49997    i am a catholic taught in parochial elementary...
49998    im going to have to disagree with the previous...
49999    no one expects the star trek movies to be high...
Name: review, Length: 50000, dtype: object
0        [27, 4, 1, 77, 1913, 42, 1050, 11, 100, 144, 3...
1        [3, 381, 114, 355, 1, 1337, 2959, 6, 50, 0, 50...
2        [9, 192, 10, 12, 3, 381, 96, 5, 1097, 58, 19, ...
3        [680, 222, 3, 235, 112, 3, 114, 434, 3562, 117...
4        [0, 0, 109, 7, 1, 58, 4, 290, 6, 3, 2124, 1355...
             

**Train-Test Split**

In [ ]:
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})


X = torch.tensor(df['padded'].tolist())
y = torch.tensor(df['sentiment'].values)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2
)

**RNN Model**

In [ ]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        out = self.fc(hidden.squeeze(0))
        return self.sigmoid(out)

In [ ]:
# Initialize Model
VOCAB_SIZE = len(word_to_idx) + 1
EMBED_DIM = 128
HIDDEN_DIM = 128

model = RNNModel(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM)

# Loss & Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

**Training Loop**

In [ ]:
# EPOCHS = 3

# for epoch in range(EPOCHS):
#     model.train()

#     y_pred = model(X_train).squeeze()
#     loss = criterion(y_pred, y_train.float())

#     optimizer.zero_grad()
#     loss.backward()
#     optimizer.step()

#     print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

EPOCHS = 3

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for batch_X, batch_y in train_loader:
        y_pred = model(batch_X).squeeze()
        loss = criterion(y_pred, batch_y.float())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

Epoch 1, Loss: 0.6977
Epoch 2, Loss: 0.6904
Epoch 3, Loss: 0.6901


**Evaluation**

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test).squeeze()
    preds = (preds >= 0.5).int()

    accuracy = (preds == y_test).float().mean()
    print("Test Accuracy:", accuracy.item())

Test Accuracy: 0.5076000094413757
